<a href="https://colab.research.google.com/github/luciazarpe/TFM/blob/main/modelizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TFM DSMarket — Modelización**

Notebook de modelización para la predicción de ventas a 28 días

**Input:** `df_preprocessed.feather` — generado por `preprocesamiento.ipynb`  
**Output:** predicciones diarias de ventas a nivel producto × tienda para los próximos 28 días

---

## **Tabla de contenidos**

1. Carga de datos
2. Feature engineering
   - 2.1 Variables de calendario
   - 2.2 Integración de clusters
   - 2.3 Rolling features y variables exógenas
3. Split temporal train/test
4. Benchmark
5. Modelo con skforecast
   - 5.1 Regresor Tweedie multiserie
   - 5.2 Optimización de hiperparámetros con Optuna
   - 5.3 Entrenamiento y predicción por tienda
   - 5.4 Feature Importances
6. Agregación bottom-up
   - 6.1 Visualizaciones de negocio
   - 6.2 Verificación de consistencia
7. Evaluación vs Benchmark
   - 7.1 Métricas por nivel jerárquico
   - 7.2 Comparativa por tienda
   - 7.3 Predicción vs real — tienda y producto
8. Exportación de resultados
9. Predicciones futuras
   - 9.1 Features exógenas para el período futuro
   - 9.2 Predicción
   - 9.3 Visualización
10. Conclusiones

---
## **0. Librerías e inicialización**

- `pandas` / `numpy` — manipulación de datos
- `plotly` — visualizaciones interactivas
- `xgboost` — regresor de ventas
- `skforecast` — framework de forecasting multiserie con soporte nativo de lags
- `sklearn` — métricas de evaluación
- `dateutil` — cálculo dinámico de Easter por año
- `joblib` — serialización de forecasters entrenados

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import gc
warnings.filterwarnings('ignore')
import json
from dateutil.easter import easter as compute_easter
import joblib

import xgboost as xgb

from skforecast.recursive import ForecasterRecursiveMultiSeries
from sklearn.preprocessing import FunctionTransformer

from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, f1_score, classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, recall_score, precision_score
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 42

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'

Mounted at /content/drive


---
## **1. Carga de datos**

Cargamos el feather generado en preprocesamiento. Ya incluye ventas diarias por producto y tienda, precios, festivos, estacionalidad e índices estacionales calculados.


In [5]:
df = pd.read_feather(DATA_PATH + 'df_preprocessed.feather')
df['date'] = pd.to_datetime(df['date'])

print(f'Shape completo: {df.shape}')
print(f'Tiendas: {df["store_code"].nunique()} → {sorted(df["store_code"].unique())}')
print(f'Productos únicos: {df["item"].nunique()}')
print(f'Rango de fechas: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\nMemoria RAM usada: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB')

Shape completo: (46027957, 22)
Tiendas: 10 → ['BOS_1', 'BOS_2', 'BOS_3', 'NYC_1', 'NYC_2', 'NYC_3', 'NYC_4', 'PHI_1', 'PHI_2', 'PHI_3']
Productos únicos: 3049
Rango de fechas: 2011-01-29 → 2016-04-24

Memoria RAM usada: 18.30 GB


---
## **2. Feature engineering**

Construimos las variables adicionales que el modelo necesita para predecir.

**Regla fundamental — sin data leakage:** cada feature calculada para el día `t` solo puede usar información de días anteriores a `t`. Si usamos datos del futuro, el modelo aprende trampeando y fallará en producción.

Las features se dividen en tres grupos:

- **Del preprocesamiento:** `sell_price`, `is_holiday`, `season`, `pay_period`, `indice_estacional_store_item`
- **De calendario:** `dayofweek`, `is_weekend`, `month`, `days_to_holiday`, `dias_desde_inicio`...
- **Basadas en ventas históricas:** rolling means, desviación típica y snapshots, siempre con un offset ≥28 días para que nunca usen predicciones propias durante el horizonte de predicción

### **2.1 Variables de calendario**

Extraemos información de la fecha que el modelo puede usar tanto en entrenamiento como en predicción: siempre sabremos qué día de la semana o a cuántos días de un festivo estará cualquier fecha futura.

- `dias_desde_inicio` captura la tendencia creciente sostenida que el EDA mostró entre 2011 y 2016.
- `days_to_holiday` captura el efecto pre-evento detectado en Thanksgiving, Easter y SuperBowl.

In [6]:
# NewYear, SuperBowl, IndependenceDay, Thanksgiving, Christmas
HOLIDAYS_FIXED_MMDD = ['01-01', '02-07', '07-04', '11-24', '12-25']

def add_calendar_features(df):
    df = df.copy()

    df['dayofweek']       = df['date'].dt.dayofweek
    df['dayofmonth']      = df['date'].dt.day
    df['weekofyear']      = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend']      = (df['date'].dt.dayofweek >= 5).astype(int)
    df['dias_desde_inicio'] = (df['date'] - df['date'].min()).dt.days

    fechas_unicas = df['date'].drop_duplicates().sort_values()
    years = range(df['date'].dt.year.min(), df['date'].dt.year.max() + 2)

    all_holidays = pd.DatetimeIndex(sorted([
        *[pd.Timestamp(f'{year}-{mmdd}') for year in years for mmdd in HOLIDAYS_FIXED_MMDD],
        *[pd.Timestamp(compute_easter(year)) for year in years]
    ]))

    dias_hasta = []
    for fecha in fechas_unicas:
        futuros = all_holidays[all_holidays >= fecha]
        dias_hasta.append((futuros[0] - fecha).days if len(futuros) > 0 else 0)

    df['days_to_holiday'] = df['date'].map(dict(zip(fechas_unicas, dias_hasta)))
    return df

df = add_calendar_features(df)

### **2.2 Integración de clusters**

El cluster de producto obtenido en el notebook de clustering se incorpora aquí como variable exógena. Agrupa productos con comportamiento similar (Core, Premium, Esporádicos, Intermedios), lo que permite al modelo aprender patrones compartidos entre productos del mismo grupo.

In [7]:
df_clusters = pd.read_csv(DATA_PATH + 'clustering_productos.csv', index_col=0)
df = df.merge(df_clusters[['item', 'cluster_kmeans']], on='item', how='left')
df['cluster_kmeans'] = df['cluster_kmeans'].astype('category')
print(f'Clusters integrados. Distribución:\n{df["cluster_kmeans"].value_counts()}')

Clusters integrados. Distribución:
cluster_kmeans
0    16304694
1    15043140
2     8067467
3     6612656
Name: count, dtype: int64


### **2.3 Rolling features y variables exógenas**

Construimos dos tipos de features basadas en ventas históricas:

- **Rolling features lagged ≥28** (`rmean_7/14/28_lag28`, `rstd_7_lag28`): medias y desviación móvil calculadas con un desfase de 28 días. Al estar siempre fuera del horizonte de predicción, nunca se recalculan con predicciones propias.
- **Snapshots de momentum** (`snap_lag7/14/21`): ventas reales de 7, 14 y 21 días antes del inicio del horizonte. Capturan el estado reciente de la serie sin riesgo de recursión.
- **Rolling a nivel tienda** (`store_rmean_7/28_lag28`): señal agregada de la tienda, menos ruidosa que la individual.

Además el **target encoding** (`te_item`, `te_store_code`) sustituye los identificadores categóricos por la mediana de ventas — calculado solo sobre train para evitar leakage.

In [8]:
# Reducir tipos para liberar RAM antes del split
df['sales'] = df['sales'].astype(np.float32)
df['sell_price'] = df['sell_price'].astype(np.float32)
df['indice_estacional_store_item'] = df['indice_estacional_store_item'].astype(np.float32)
df['dias_desde_inicio'] = df['dias_desde_inicio'].astype(np.int16)
df['days_to_holiday'] = df['days_to_holiday'].astype(np.int16)

print(f'Memoria tras optimización: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Memoria tras optimización: 24.05 GB


In [9]:
# Rolling features manuales lagged ≥28 — nunca usan predicciones propias

# A nivel producto-tienda (id)
for window in [7, 14, 28]:
    df[f'rmean_{window}_lag28'] = (
        df.groupby('id')['sales']
        .transform(lambda x, w=window: x.shift(28).rolling(w).mean())
        .astype(np.float32)
    )

df['rstd_7_lag28'] = (
    df.groupby('id')['sales']
    .transform(lambda x: x.shift(28).rolling(7).std())
    .astype(np.float32)
)

# Snapshots estáticos — momentum justo antes del corte
for lag in [7, 14, 21]:
    df[f'snap_lag{lag}'] = (
        df.groupby('id')['sales']
        .transform(lambda x, l=lag: x.shift(l))
        .astype(np.float32)
    )

# A nivel tienda — señal agregada, menos ruido
store_daily = (
    df.groupby(['store_code', 'date'])['sales']
    .mean().reset_index(name='store_sales_mean')
)
for window in [7, 28]:
    store_daily[f'store_rmean_{window}_lag28'] = (
        store_daily.groupby('store_code')['store_sales_mean']
        .transform(lambda x, w=window: x.shift(28).rolling(w).mean())
        .astype(np.float32)
    )
df = df.merge(
    store_daily[['store_code', 'date', 'store_rmean_7_lag28', 'store_rmean_28_lag28']],
    on=['store_code', 'date'], how='left'
)

print('Features OK')
print(df[['rmean_7_lag28','rmean_14_lag28','rmean_28_lag28',
          'snap_lag7','snap_lag14','snap_lag21',
          'store_rmean_7_lag28','store_rmean_28_lag28']].describe())

Features OK
       rmean_7_lag28  rmean_14_lag28  rmean_28_lag28     snap_lag7  \
count   4.499130e+07    4.477787e+07    4.435101e+07  4.581453e+07   
mean    1.428274e+00    1.428153e+00    1.427786e+00  1.427713e+00   
std     3.638454e+00    3.557938e+00    3.492121e+00  4.142580e+00   
min     0.000000e+00    0.000000e+00    0.000000e+00  0.000000e+00   
25%     1.428571e-01    1.428571e-01    1.785714e-01  0.000000e+00   
50%     4.285714e-01    5.000000e-01    5.357143e-01  0.000000e+00   
75%     1.428571e+00    1.357143e+00    1.357143e+00  1.000000e+00   
max     6.028571e+02    4.686429e+02    4.427500e+02  7.630000e+02   

         snap_lag14    snap_lag21  store_rmean_7_lag28  store_rmean_28_lag28  
count  4.560110e+07  4.538767e+07         4.561218e+07          4.532778e+07  
mean   1.427906e+00  1.427776e+00         1.428517e+00          1.426655e+00  
std    4.145585e+00  4.148177e+00         6.549800e-01          6.192480e-01  
min    0.000000e+00  0.000000e+00        

In [10]:
EXOG_FEATURES_BASE = [
    'sell_price', 'is_holiday', 'indice_estacional_store_item',
    'season', 'pay_period',
    'dayofweek', 'dayofmonth', 'weekofyear', 'month', 'year',
    'is_weekend', 'dias_desde_inicio', 'days_to_holiday',
    'cluster_kmeans'
]

EXOG_FEATURES = EXOG_FEATURES_BASE + [
    'te_store_code', 'te_item',
    'rmean_7_lag28', 'rmean_14_lag28', 'rmean_28_lag28', 'rstd_7_lag28',
    'snap_lag7', 'snap_lag14', 'snap_lag21',
    'store_rmean_7_lag28', 'store_rmean_28_lag28'
]

TARGET = 'sales'
print(f'Features regresor ({len(EXOG_FEATURES)}): {EXOG_FEATURES}')

Features regresor (25): ['sell_price', 'is_holiday', 'indice_estacional_store_item', 'season', 'pay_period', 'dayofweek', 'dayofmonth', 'weekofyear', 'month', 'year', 'is_weekend', 'dias_desde_inicio', 'days_to_holiday', 'cluster_kmeans', 'te_store_code', 'te_item', 'rmean_7_lag28', 'rmean_14_lag28', 'rmean_28_lag28', 'rstd_7_lag28', 'snap_lag7', 'snap_lag14', 'snap_lag21', 'store_rmean_7_lag28', 'store_rmean_28_lag28']


---
## **3. Split temporal train/test**

Reservamos los **últimos 28 días** como conjunto de test — el mismo horizonte de predicción que pide el enunciado.

En series temporales el split **siempre es temporal**, nunca aleatorio. Un split aleatorio introduciría data leakage: el modelo vería datos del futuro durante el entrenamiento y daría métricas artificialmente buenas que no se replicarían en producción.

Después del split liberamos el dataframe completo de memoria (`del df`) para no saturar la RAM en Colab.

In [11]:
fecha_corte = df['date'].max() - pd.Timedelta(days=28)
print(f'Fecha de corte: {fecha_corte}')

Fecha de corte: 2016-03-27 00:00:00


In [12]:
df_train = df[df['date'] <= fecha_corte].copy()
df_test  = df[df['date'] >  fecha_corte].copy()

del df
gc.collect()

print(f'Train: {df_train["date"].min().date()} → {df_train["date"].max().date()} ({df_train["date"].nunique()} días)')
print(f'Test:  {df_test["date"].min().date()} → {df_test["date"].max().date()} ({df_test["date"].nunique()} días)')

Train: 2011-01-29 → 2016-03-27 (1885 días)
Test:  2016-03-28 → 2016-04-24 (28 días)


In [13]:
# Target encoding — solo sobre train para evitar data leakage!!
for col in ['store_code', 'item']:
    mediana = df_train.groupby(col)['sales'].median()
    df_train[f'te_{col}'] = df_train[col].map(mediana).astype(np.float32)
    df_test[f'te_{col}']  = df_test[col].map(mediana).astype(np.float32)
    print(f'te_{col}: min={df_train[f"te_{col}"].min():.2f}, max={df_train[f"te_{col}"].max():.2f}')

te_store_code: min=0.00, max=1.00
te_item: min=0.00, max=42.00


---
## **4. Benchmark**

Antes de entrenar cualquier modelo de ML necesitamos un **punto de referencia** contra el que medir si nuestro modelo aporta valor real.

El benchmark replica el enfoque actual de DSMarket: para cada serie (producto × tienda) toma la **forma** de las ventas del mismo período del año anterior y la escala por el **nivel reciente** de ventas (media de los últimos 28 días). Es un modelo estacional simple, sin ML.

Si nuestro modelo no supera este benchmark, no estamos aportando valor.

In [21]:
def benchmark_prediccion(df_train, df_test):

    # Nivel reciente: media de los últimos 28 días por serie
    nivel_reciente = (
        df_train.sort_values('date')
        .groupby('id')
        .tail(28)
        .groupby('id')['sales']
        .mean()
        .rename('nivel_reciente')
    )

    # Forma del año anterior: mismo período 365 días antes
    fecha_ini = df_test['date'].min()
    fecha_fin = df_test['date'].max()

    mismo_periodo = df_train[
        (df_train['date'] >= fecha_ini - pd.Timedelta(days=365)) &
        (df_train['date'] <= fecha_fin - pd.Timedelta(days=365))
    ].copy()
    mismo_periodo['date'] = mismo_periodo['date'] + pd.Timedelta(days=365)

    nivel_anterior = (
        mismo_periodo.groupby('id')['sales']
        .mean()
        .rename('nivel_anterior')
    )

    # Merge todo junto
    df_bench = df_test[['id', 'date', 'sales']].copy()
    df_bench = df_bench.merge(mismo_periodo[['id', 'date', 'sales']].rename(columns={'sales': 'forma_anterior'}), on=['id', 'date'], how='left')
    df_bench = df_bench.merge(nivel_reciente, on='id', how='left')
    df_bench = df_bench.merge(nivel_anterior, on='id', how='left')

    # Escalar la forma del año anterior al nivel reciente
    df_bench['nivel_anterior'] = df_bench['nivel_anterior'].replace(0, np.nan)
    df_bench['pred_bench'] = (
        df_bench['forma_anterior'] * (df_bench['nivel_reciente'] / df_bench['nivel_anterior'])
    ).clip(lower=0)

    # Fallback: si no hay año anterior, usar nivel reciente
    df_bench['pred_bench'] = df_bench['pred_bench'].fillna(df_bench['nivel_reciente'])

    return df_bench[['id', 'date', 'sales', 'pred_bench']].rename(columns={'sales': 'real'})

df_benchmark = benchmark_prediccion(df_train, df_test)

Calculamos las tres métricas que usaremos a lo largo de todo el notebook:

- **MAE** (Mean Absolute Error): error medio en unidades. Fácil de interpretar para el negocio.
- **RMSE** (Root Mean Squared Error): penaliza más los errores grandes. Sensible a picos.
- **WMAPE** (Weighted MAPE): error porcentual ponderado por volumen. Permite comparar entre series de distinto tamaño y es la métrica más usada en retail forecasting.

In [22]:
mae_bench  = mean_absolute_error(df_benchmark['real'], df_benchmark['pred_bench'])
rmse_bench = root_mean_squared_error(df_benchmark['real'], df_benchmark['pred_bench'])

def wmape(real, pred):
    return np.sum(np.abs(real - pred)) / np.sum(np.abs(real))

wmape_bench = wmape(df_benchmark['real'].values, df_benchmark['pred_bench'].values)

print(f'Benchmark — MAE:   {mae_bench:.4f}')
print(f'Benchmark — RMSE:  {rmse_bench:.4f}')
print(f'Benchmark — WMAPE: {wmape_bench*100:.2f}%')

Benchmark — MAE:   1.3808
Benchmark — RMSE:  3.8115
Benchmark — WMAPE: 99.60%


---
## **5. Modelo con skforecast**

Entrenamos un regresor independiente por tienda usando `ForecasterRecursiveMultiSeries` de skforecast. Cada tienda genera su propio modelo, capturando patrones específicos de cada ubicación sin interferencia entre tiendas.

### **Funciones auxiliares**

- `preparar_exog`: convierte columnas string a categoría para XGBoost con **enable_categorical=True**
- `preparar_datos_tienda`: pivota el dataframe largo a formato wide (fecha × producto) que requiere skforecast, y construye el diccionario de exog separando features de tienda de features de producto
- `entrenar_regresor`: inicializa y entrena el **ForecasterRecursiveMultiSeries** con los hiperparámetros optimizados por Optuna

In [14]:
def preparar_exog(df, exog_features):
    df = df.copy()
    for col in exog_features:
        if df[col].dtype == 'object':
            df[col] = df[col].astype('category')
    return df

In [15]:
def preparar_datos_tienda(df, store_code, exog_features, target='sales'):
    df_tienda = df[df['store_code'] == store_code].copy()

    # Categorías globales — iguales para todos los productos
    season_cats    = df_train['season'].dropna().unique().tolist()
    pay_cats       = df_train['pay_period'].dropna().unique().tolist()

    # Formato wide: índice=fecha, columnas=productos
    series = df_tienda.pivot(index='date', columns='item', values=target)
    series.index = pd.DatetimeIndex(series.index, freq='D')

    # Features de tienda — iguales para todos los productos
    exog_store_features = [
        'is_holiday', 'dayofweek', 'dayofmonth', 'weekofyear',
        'month', 'year', 'is_weekend', 'dias_desde_inicio', 'days_to_holiday',
        'store_rmean_7_lag28', 'store_rmean_28_lag28'
    ]

    # Features de producto — varían por item dentro de la tienda
    exog_product_features = [
        'sell_price', 'indice_estacional_store_item', 'season', 'pay_period',
        'te_store_code', 'te_item',
        'rmean_7_lag28', 'rmean_14_lag28', 'rmean_28_lag28', 'rstd_7_lag28',
        'snap_lag7', 'snap_lag14', 'snap_lag21'
    ]

    # Diccionario: una entrada por producto con todas las features
    exog_dict = {}
    for item in df_tienda['item'].unique():
        df_item = df_tienda[df_tienda['item'] == item].set_index('date')
        exog_item = df_item[exog_store_features + exog_product_features].copy()

        exog_item['season']     = pd.Categorical(exog_item['season'],     categories=season_cats)
        exog_item['pay_period'] = pd.Categorical(exog_item['pay_period'], categories=pay_cats)

        exog_item.index = pd.DatetimeIndex(exog_item.index, freq='D')
        exog_dict[item] = exog_item

    return series, exog_dict

### **5.1 Regresor Tweedie multiserie**

`ForecasterRecursiveMultiSeries` con `XGBRegressor` entrenado por tienda.

**Objetivo Tweedie** (`reg:tweedie`, `variance_power=1.1664`): diseñado para datos de conteo con muchos ceros — exactamente el perfil de la demanda intermitente en retail. Los hiperparámetros finales (**n_estimators=500**, **max_depth=7**, **min_child_weight=19**, **learning_rate=0.0816**) se obtuvieron mediante búsqueda bayesiana con **Optuna** (30 trials sobre NYC_1), consiguiendo un MAE de 1.008 en esa tienda frente a 1.38 del benchmark.

**Lags seguros:** [28, 35, 42] días. En un horizonte de 28 días, estos lags siempre apuntan a datos reales de train — nunca a predicciones propias.

In [17]:
def entrenar_regresor(series_train, exog_train, lags=[28, 35, 42]):
    forecaster = ForecasterRecursiveMultiSeries(
        estimator = xgb.XGBRegressor(
            objective              = 'reg:tweedie',
            tweedie_variance_power = 1.1664,
            n_estimators           = 500,
            learning_rate          = 0.0816,
            max_depth              = 7,
            subsample              = 0.9579,
            colsample_bytree       = 0.8392,
            min_child_weight       = 19,
            enable_categorical     = True,
            random_state           = RANDOM_SEED,
            n_jobs                 = -1,
            verbosity              = 0
        ),
        lags     = lags,
        encoding = 'ordinal_category'
    )
    forecaster.fit(series=series_train, exog=exog_train)
    return forecaster

### **5.2 Optimización de hiperparámetros con Optuna**

Los hiperparámetros del regresor se obtuvieron mediante búsqueda bayesiana con **Optuna** (30 trials sobre NYC_1). El espacio de búsqueda cubrió `tweedie_variance_power`, `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `colsample_bytree` y `min_child_weight`.

El mejor trial alcanzó un **MAE de 1.008** en NYC_1 frente a 1.38 del benchmark — una mejora del 27% en una sola tienda. Los parámetros resultantes se fijaron en `entrenar_regresor` y se aplicaron a las 10 tiendas.

In [ ]:
# Búsqueda de hiperparámetros con Optuna — 30 trials sobre NYC_1
# Ejecutado en notebook aparte (optuna.ipynb)

# import optuna
# from optuna.samplers import TPESampler
# study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_SEED))
# study.optimize(objective, n_trials=30, show_progress_bar=True)

# --- RESULTADO ---
# Mejor MAE: 1.0080  (benchmark NYC_1: 1.3808)
#
# Mejores params:
#   tweedie_variance_power : 1.1664
#   n_estimators           : 500
#   learning_rate          : 0.0816
#   max_depth              : 7
#   subsample              : 0.9579
#   colsample_bytree       : 0.8392
#   min_child_weight       : 19
#
# → Integrados en entrenar_regresor()

### **5.3 Entrenamiento y predicción por tienda**

Bucle sobre las 10 tiendas: para cada una se preparan series y exog, se filtra por mínimo de histórico (56 días), se entrena el regresor y se generan predicciones a 28 días.

`preparar datos → entrenar regresor → predecir 28 días → guardar resultados`


In [18]:
resultados_modelo = []
stores = sorted(df_train['store_code'].unique())

for store in stores:
    print(f'\n→ Procesando tienda: {store}')

    series_train, exog_train = preparar_datos_tienda(df_train, store, EXOG_FEATURES)
    series_test,  exog_test  = preparar_datos_tienda(df_test,  store, EXOG_FEATURES)

    series_con_ventas = series_train.columns[series_train.sum() > 0]
    series_train_fil  = series_train[series_con_ventas]
    exog_train_fil    = {k: v for k, v in exog_train.items() if k in series_con_ventas}
    exog_test_fil     = {k: v for k, v in exog_test.items()  if k in series_con_ventas}

    min_dias = 56
    series_validas = series_train_fil.columns[series_train_fil.notna().sum() >= min_dias]
    n_excluidas = len(series_train_fil.columns) - len(series_validas)
    print(f'  Series excluidas por histórico corto: {n_excluidas}')
    series_train_fil = series_train_fil[series_validas]
    exog_train_fil   = {k: v for k, v in exog_train_fil.items() if k in series_validas}
    exog_test_fil    = {k: v for k, v in exog_test_fil.items()  if k in series_validas}

    forecaster = entrenar_regresor(series_train_fil, exog_train_fil)
    print(f'  ✅ Regresor {store} OK — {len(series_validas)} series')
    joblib.dump(forecaster, DATA_PATH + f'forecaster_{store}.pkl')

    pred      = forecaster.predict(steps=28, exog=exog_test_fil)
    pred_long = pred.reset_index()
    pred_long.columns = ['date', 'item', 'pred_final']
    pred_long['pred_final'] = pred_long['pred_final'].clip(lower=0).round(2)

    df_store = df_test[df_test['store_code'] == store][['date', 'store_code', 'item', 'sales']].copy()
    df_store = df_store.merge(pred_long, on=['date', 'item'], how='left')
    df_store['pred_final'] = df_store['pred_final'].fillna(0)

    resultados_modelo.append(df_store)
    pd.concat(resultados_modelo).to_csv(DATA_PATH + f'resultados_parcial_{store}.csv', index=False)
    print(f'  ✅ {store} completado y guardado')

df_resultados = pd.concat(resultados_modelo).reset_index(drop=True)
print(f'\nShape final: {df_resultados.shape}')
print(df_resultados.head())


→ Procesando tienda: BOS_1
  Series excluidas por histórico corto: 0
  ✅ Regresor BOS_1 OK — 3049 series
  ✅ BOS_1 completado y guardado

→ Procesando tienda: BOS_2
  Series excluidas por histórico corto: 0
  ✅ Regresor BOS_2 OK — 3049 series
  ✅ BOS_2 completado y guardado

→ Procesando tienda: BOS_3
  Series excluidas por histórico corto: 0
  ✅ Regresor BOS_3 OK — 3049 series
  ✅ BOS_3 completado y guardado

→ Procesando tienda: NYC_1
  Series excluidas por histórico corto: 1
  ✅ Regresor NYC_1 OK — 3048 series
  ✅ NYC_1 completado y guardado

→ Procesando tienda: NYC_2
  Series excluidas por histórico corto: 2
  ✅ Regresor NYC_2 OK — 3047 series
  ✅ NYC_2 completado y guardado

→ Procesando tienda: NYC_3
  Series excluidas por histórico corto: 2
  ✅ Regresor NYC_3 OK — 3047 series
  ✅ NYC_3 completado y guardado

→ Procesando tienda: NYC_4
  Series excluidas por histórico corto: 1
  ✅ Regresor NYC_4 OK — 3048 series
  ✅ NYC_4 completado y guardado

→ Procesando tienda: PHI_1
  Seri

Guardamos los resultados para no tener que reentrenar si se reinicia la sesión. La celda de carga permite retomar el notebook desde aquí directamente.

In [34]:
# Guardar resultados
df_resultados.to_csv(DATA_PATH + 'resultados_modelo.csv', index=False)
print('✅ Resultados guardados.')

✅ Resultados guardados.


In [16]:
# Cargar resultados
df_resultados = pd.read_csv(DATA_PATH + 'resultados_modelo.csv', parse_dates=['date'])
print('✅ Resultados cargados.')
print(df_resultados.shape)
print(df_resultados.head())

✅ Resultados cargados.
(853720, 5)
        date store_code              item  sales  pred_final
0 2016-03-28      BOS_1  ACCESORIES_1_001    1.0        0.29
1 2016-03-28      BOS_1  ACCESORIES_1_002    0.0        0.01
2 2016-03-28      BOS_1  ACCESORIES_1_003    0.0        0.01
3 2016-03-28      BOS_1  ACCESORIES_1_004    0.0        0.78
4 2016-03-28      BOS_1  ACCESORIES_1_005    0.0        0.30


### **5.4 Feature Importances**

Analizamos qué variables tienen más peso en el modelo entrenado sobre la última tienda del bucle.

In [36]:
# Feature importance
importance = forecaster.get_feature_importances().sort_values('importance', ascending=False)
fig = px.bar(importance.head(20), x='importance', y='feature',
             orientation='h',
             title=f'Top 20 features — Variables importantes para el modelo',
             template='plotly_white')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## **6. Agregación bottom-up**

Nos piden predicciones a nivel producto × tienda. Para obtener totales por tienda, ciudad o cadena completa, **sumamos las predicciones individuales hacia arriba** (bottom-up).

Este es exactamente el enfoque que describe Paul en su email: predecir a nivel tienda-producto y agregar sumando para obtener ventas por departamento, tienda o ciudad.

Construimos cuatro niveles:

| Nivel | Descripción | Filas |
|---|---|---|
| Producto × Tienda | Serie individual por producto y tienda | ~854k |
| Tienda × Fecha | Suma de todos los productos de cada tienda | 280 |
| Ciudad × Fecha | Suma de tiendas por ciudad | 84 |
| Total DSMarket | Ventas totales de la cadena | 28 |

In [37]:
# Nivel 0: detalle (ya existe, solo limpiamos columnas útiles)
df_nivel_item = df_resultados[['date', 'store_code', 'item', 'sales', 'pred_final']].copy()

# Nivel 1: tienda × fecha
df_nivel_tienda = (
    df_nivel_item
    .groupby(['date', 'store_code'], observed=True)
    .agg(sales=('sales', 'sum'), pred_final=('pred_final', 'sum'))
    .reset_index()
)

# Nivel 2: ciudad × fecha (necesitamos el mapeo store → ciudad)
store_ciudad = {
    'NYC_1': 'New York', 'NYC_2': 'New York', 'NYC_3': 'New York', 'NYC_4': 'New York',
    'BOS_1': 'Boston',   'BOS_2': 'Boston',   'BOS_3': 'Boston',
    'PHI_1': 'Philadelphia', 'PHI_2': 'Philadelphia', 'PHI_3': 'Philadelphia'
}
df_nivel_tienda['city'] = df_nivel_tienda['store_code'].map(store_ciudad)

df_nivel_ciudad = (
    df_nivel_tienda
    .groupby(['date', 'city'])
    .agg(sales=('sales', 'sum'), pred_final=('pred_final', 'sum'))
    .reset_index()
)

# Nivel 3: total × fecha
df_nivel_total = (
    df_nivel_tienda
    .groupby('date')
    .agg(sales=('sales', 'sum'), pred_final=('pred_final', 'sum'))
    .reset_index()
)
df_nivel_total['nivel'] = 'Total DSMarket'

print('Niveles de agregación construidos:')
print(f'  Item × Tienda:  {df_nivel_item.shape[0]:,} filas')
print(f'  Tienda:         {df_nivel_tienda.shape[0]:,} filas')
print(f'  Ciudad:         {df_nivel_ciudad.shape[0]:,} filas')
print(f'  Total:          {df_nivel_total.shape[0]:,} filas')

Niveles de agregación construidos:
  Item × Tienda:  853,720 filas
  Tienda:         280 filas
  Ciudad:         84 filas
  Total:          28 filas


### **6.1 Visualizaciones de negocio**

Tres vistas en cascada, de lo más general a lo más específico:
1. **Total DSMarket** — para el director financiero (Paul)
2. **Por ciudad** — para la CDO (Michelle) y el equipo de marketing
3. **Por tienda** — para operaciones

In [23]:
# --- 1. Total DSMarket ---
fig_total = go.Figure()

fig_total.add_trace(go.Scatter(
    x=df_nivel_total['date'], y=df_nivel_total['sales'],
    name='Real', line=dict(color='#2ca02c', width=2.5)
))
fig_total.add_trace(go.Scatter(
    x=df_nivel_total['date'], y=df_nivel_total['pred_final'],
    name='Predicción', line=dict(color='#1f77b4', width=2.5, dash='dash')
))
fig_total.update_layout(
    title='Ventas totales DSMarket — 28 días',
    xaxis_title='Fecha', yaxis_title='Unidades vendidas',
    hovermode='x unified', template='plotly_white'
)
fig_total.show()

# --- 2. Por ciudad ---
df_ciudad_plot = df_nivel_ciudad.melt(
    id_vars=['date', 'city'],
    value_vars=['sales', 'pred_final'],
    var_name='tipo', value_name='unidades'
)
df_ciudad_plot['tipo'] = df_ciudad_plot['tipo'].map({'sales': 'Real', 'pred_final': 'Predicción'})
df_ciudad_plot['serie'] = df_ciudad_plot['city'] + ' — ' + df_ciudad_plot['tipo']

fig_ciudad = px.line(
    df_ciudad_plot,
    x='date', y='unidades', color='city', line_dash='tipo',
    line_dash_map={'Real': 'solid', 'Predicción': 'dash'},
    labels={'date': 'Fecha', 'unidades': 'Unidades', 'city': 'Ciudad', 'tipo': ''},
    title='Ventas por ciudad — 28 días',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_ciudad.update_layout(hovermode='x unified')
fig_ciudad.show()

# --- 3. Por tienda ---
df_tienda_plot = df_nivel_tienda.melt(
    id_vars=['date', 'store_code'],
    value_vars=['sales', 'pred_final'],
    var_name='tipo', value_name='unidades'
)
df_tienda_plot['tipo'] = df_tienda_plot['tipo'].map({'sales': 'Real', 'pred_final': 'Predicción'})

fig_tienda = px.line(
    df_tienda_plot,
    x='date', y='unidades', color='store_code', line_dash='tipo',
    line_dash_map={'Real': 'solid', 'Predicción': 'dash'},
    labels={'date': 'Fecha', 'unidades': 'Unidades', 'store_code': 'Tienda', 'tipo': ''},
    title='Ventas por tienda — 28 días',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig_tienda.update_layout(hovermode='x unified')
fig_tienda.show()

### **6.2 Verificación de consistencia**

Comprobamos que la jerarquía cuadra: la suma de predicciones por ciudad debe ser igual
al total, y la suma por tienda debe ser igual a la suma por ciudad.
Esta verificación es obligatoria antes de exportar — un error aquí invalidaría
cualquier análisis downstream.

In [38]:
# Verificación: la suma de ciudades debe coincidir con el total
suma_ciudades = df_nivel_ciudad.groupby('date')['sales'].sum()
suma_total = df_nivel_total.set_index('date')['sales']

assert (suma_ciudades == suma_total).all(), "❌ Las sumas no cuadran"
print("✅ Consistencia verificada: suma de ciudades = total DSMarket")

# Ídem predicciones
suma_pred_ciudades = df_nivel_ciudad.groupby('date')['pred_final'].sum()
suma_pred_total = df_nivel_total.set_index('date')['pred_final']
assert (suma_pred_ciudades.round(4) == suma_pred_total.round(4)).all(), "❌ Predicciones no cuadran"
print("✅ Consistencia verificada: suma de predicciones por ciudad = total")

✅ Consistencia verificada: suma de ciudades = total DSMarket
✅ Consistencia verificada: suma de predicciones por ciudad = total


---
## **7. Evaluación vs Benchmark**

Evaluamos el modelo con métricas numéricas en todos los niveles jerárquicos y con visualizaciones de las predicciones frente a los valores reales y el benchmark.

### **7.1 Métricas por nivel jerárquico**

Calculamos WMAPE y MAE para benchmark y modelo en cada nivel de la jerarquía — desde el producto individual hasta el total de la cadena.

In [39]:
# Preparamos modelo benchmark para poder agregar por cualquier nivel
df_meta = df_train[['store_code', 'item', 'category', 'region']].drop_duplicates()
df_resultados_ext = df_resultados.merge(df_meta, on=['store_code', 'item'], how='left')

bench_ext = df_benchmark.copy()
bench_ext['store_code'] = bench_ext['id'].str.extract(r'([A-Z]+_\d+)$')
bench_ext['item']       = bench_ext['id'].str.replace(r'_[A-Z]+_\d+$', '', regex=True)
bench_ext = bench_ext.merge(df_meta, on=['store_code', 'item'], how='left')
bench_ext = bench_ext.rename(columns={'real': 'sales', 'pred_bench': 'pred_final'})

print(f'Modelo:    {len(df_resultados_ext):,} filas')
print(f'Benchmark: {len(bench_ext):,} filas')
print(f'NaN en benchmark')

Modelo:    853,720 filas
Benchmark: 853,720 filas
NaN en benchmark


In [40]:
def comp(df_m, df_b, nivel):
    wm_m  = wmape(df_m['sales'].values, df_m['pred_final'].values) * 100
    wm_b  = wmape(df_b['sales'].values, df_b['pred_final'].values) * 100
    mae_m = mean_absolute_error(df_m['sales'], df_m['pred_final'])
    mae_b = mean_absolute_error(df_b['sales'], df_b['pred_final'])
    return {
        'Nivel':            nivel,
        'MAE Benchmark':    round(mae_b, 2),
        'MAE Modelo':       round(mae_m, 2),
        'Mejora MAE':       f"{(mae_b - mae_m) / mae_b * 100:.1f}%",
        'WMAPE Benchmark':  f"{wm_b:.1f}%",
        'WMAPE Modelo':     f"{wm_m:.1f}%",
        'Mejora WMAPE':     f"{(wm_b - wm_m) / wm_b * 100:.1f}%",
    }

niveles_eval = [
    ('Producto × Tienda',
     df_resultados_ext,
     bench_ext),
    ('Categoría × Tienda',
     df_resultados_ext.groupby(['date','store_code','category'])[['sales','pred_final']].sum().reset_index(),
     bench_ext.groupby(['date','store_code','category'])[['sales','pred_final']].sum().reset_index()),
    ('Tienda',
     df_resultados_ext.groupby(['date','store_code'])[['sales','pred_final']].sum().reset_index(),
     bench_ext.groupby(['date','store_code'])[['sales','pred_final']].sum().reset_index()),
    ('Ciudad',
     df_resultados_ext.groupby(['date','region'])[['sales','pred_final']].sum().reset_index(),
     bench_ext.groupby(['date','region'])[['sales','pred_final']].sum().reset_index()),
    ('Total DSMarket',
     df_resultados_ext.groupby('date')[['sales','pred_final']].sum().reset_index(),
     bench_ext.groupby('date')[['sales','pred_final']].sum().reset_index()),
]

df_resumen = pd.DataFrame([comp(m, b, n) for n, m, b in niveles_eval])
print(df_resumen.to_string(index=False))

             Nivel  MAE Benchmark  MAE Modelo Mejora MAE WMAPE Benchmark WMAPE Modelo Mejora WMAPE
 Producto × Tienda           1.38        0.94      31.8%           99.6%        67.9%        31.8%
Categoría × Tienda         230.11      137.06      40.4%           16.3%         9.7%        40.4%
            Tienda         606.19      355.92      41.3%           14.3%         8.4%        41.3%
            Ciudad        1845.19      933.20      49.4%           13.1%         6.6%        49.4%
    Total DSMarket        5319.56     2411.37      54.7%           12.6%         5.7%        54.7%


In [41]:
fig = go.Figure()
fig.add_trace(go.Bar(
    name='Benchmark',
    x=df_resumen['Nivel'],
    y=df_resumen['WMAPE Benchmark'].str.replace('%','').astype(float),
    marker_color='#666666', opacity=0.6,
    text=df_resumen['WMAPE Benchmark'], textposition='outside'
))
fig.add_trace(go.Bar(
    name='Modelo',
    x=df_resumen['Nivel'],
    y=df_resumen['WMAPE Modelo'].str.replace('%','').astype(float),
    marker_color='#2ca02c', opacity=0.85,
    text=df_resumen['WMAPE Modelo'], textposition='outside'
))
fig.update_layout(
    barmode='group',
    title='WMAPE por nivel de agregación — Modelo vs Benchmark',
    yaxis_title='WMAPE (%)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

### **7.2 Comparativa por tienda**

El modelo supera al benchmark en las 10 tiendas. Desglosamos la mejora individualmente:

In [42]:
df_comparativa_rows = []
for store in sorted(df_resultados['store_code'].unique()):
    df_m = df_resultados[df_resultados['store_code'] == store]
    df_b = bench_ext[bench_ext['store_code'] == store]
    df_comparativa_rows.append({
        'store':       store,
        'MAE':         mean_absolute_error(df_m['sales'], df_m['pred_final']),
        'MAE_bench':   mean_absolute_error(df_b['sales'], df_b['pred_final']),
        'WMAPE':       wmape(df_m['sales'].values, df_m['pred_final'].values),
        'WMAPE_bench': wmape(df_b['sales'].values, df_b['pred_final'].values),
    })
df_comparativa = pd.DataFrame(df_comparativa_rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=('MAE por tienda', 'WMAPE por tienda'))
tiendas = df_comparativa['store']
fig.add_trace(go.Bar(name='Benchmark', x=tiendas, y=df_comparativa['MAE_bench'],
    marker_color='#666666', opacity=0.6), row=1, col=1)
fig.add_trace(go.Bar(name='Modelo', x=tiendas, y=df_comparativa['MAE'],
    marker_color='#2ca02c', opacity=0.85), row=1, col=1)
fig.add_trace(go.Bar(name='Benchmark', x=tiendas, y=df_comparativa['WMAPE_bench']*100,
    marker_color='#666666', opacity=0.6, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(name='Modelo', x=tiendas, y=df_comparativa['WMAPE']*100,
    marker_color='#2ca02c', opacity=0.85, showlegend=False), row=1, col=2)
fig.update_layout(barmode='group', template='plotly_white',
    title='Error por tienda — Modelo vs Benchmark',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

### **7.3 Predicción vs real - tienda y producto**

Cada gráfica muestra 90 días de histórico + los 28 días de test, con las líneas del modelo (naranja) y el benchmark (gris). La línea vertical marca el inicio del período de test.

In [43]:
def plot_prediccion_tienda(df_train, df_resultados, df_benchmark, store, dias_contexto=90):
    # Histórico agregado de la tienda
    train_store = df_train[df_train['store_code'] == store].groupby('date')['sales'].sum().reset_index()
    train_store = train_store.tail(dias_contexto)

    # Real, modelo y benchmark agregados
    test_store = df_resultados[df_resultados['store_code'] == store].groupby('date').agg(
        real       = ('sales', 'sum'),
        pred_final = ('pred_final', 'sum')
    ).reset_index()

    bench_store = df_benchmark[df_benchmark['id'].str.endswith(store)].groupby('date').agg(
        pred_bench = ('pred_bench', 'sum')
    ).reset_index()

    test_store = test_store.merge(bench_store, on='date', how='left')

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=train_store['date'], y=train_store['sales'],
        name='Histórico', line=dict(color='#1f77b4', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=test_store['date'], y=test_store['real'],
        name='Real', line=dict(color='#2ca02c', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=test_store['date'], y=test_store['pred_final'],
        name='Modelo', line=dict(color='#ff7f0e', width=2.5, dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=test_store['date'], y=test_store['pred_bench'],
        name='Benchmark', line=dict(color='#666666', width=2, dash='dot')
    ))

    fig.add_shape(
        type='line',
        x0=str(fecha_corte.date()), x1=str(fecha_corte.date()),
        y0=0, y1=1, yref='paper',
        line=dict(color='gray', dash='dot', width=1.5)
    )
    fig.add_annotation(
        x=str(fecha_corte.date()), y=1, yref='paper',
        text='Corte train/test', showarrow=False,
        font=dict(color='gray', size=11), xanchor='left'
    )

    test_store_clean = test_store.dropna(subset=['pred_bench', 'pred_final'])

    mae_modelo  = mean_absolute_error(test_store_clean['real'], test_store_clean['pred_final'])
    mae_bench_s = mean_absolute_error(test_store_clean['real'], test_store_clean['pred_bench'])

    titulo_periodo = f"{test_store['date'].min().strftime('%d %b %Y')} → {test_store['date'].max().strftime('%d %b %Y')}"

    fig.update_layout(
        title=f'{store} — Ventas totales agregadas | {titulo_periodo}',
        xaxis_title='Fecha',
        yaxis_title='Unidades vendidas (total tienda)',
        template='plotly_white',
        height=450
    )

    fig.show()

# Una gráfica por tienda
for store in sorted(df_resultados['store_code'].unique()):
    plot_prediccion_tienda(df_train, df_resultados, df_benchmark, store)

El modelo sigue bien la serie en las primeras semanas. En los últimos días del horizonte la predicción pierde nivel en todas las tiendas — limitación inherente al forecasting recursivo a 28 días, analizada en las conclusiones.

In [44]:
for cat in ['SUPERMARKET', 'HOME_&_GARDEN', 'ACCESORIES']:
    top = df_train[
        (df_train['store_code'] == 'NYC_1') &
        (df_train['category'] == cat)
    ].groupby('item')['sales'].sum().sort_values(ascending=False)
    print(f'\n{cat}:')
    print(top.head(3))


SUPERMARKET:
item
SUPERMARKET_3_090    125780.0
SUPERMARKET_3_586     86616.0
SUPERMARKET_3_252     73920.0
Name: sales, dtype: float32

HOME_&_GARDEN:
item
HOME_&_GARDEN_1_118    14353.0
HOME_&_GARDEN_1_535    12465.0
HOME_&_GARDEN_1_334    12430.0
Name: sales, dtype: float32

ACCESORIES:
item
ACCESORIES_1_348    22559.0
ACCESORIES_1_371    22289.0
ACCESORIES_1_268    18540.0
Name: sales, dtype: float32


In [45]:
def plot_prediccion_producto(df_train, df_resultados, store, item, dias_contexto=90):
    train_serie = df_train[
        (df_train['store_code'] == store) & (df_train['item'] == item)
    ].sort_values('date').tail(dias_contexto)

    test_serie = df_resultados[
        (df_resultados['store_code'] == store) & (df_resultados['item'] == item)
    ].sort_values('date')

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=train_serie['date'], y=train_serie['sales'],
        name='Histórico', line=dict(color='#1f77b4', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=test_serie['date'], y=test_serie['sales'],
        name='Real', line=dict(color='#2ca02c', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=test_serie['date'], y=test_serie['pred_final'],
        name='Predicción', line=dict(color='#ff7f0e', width=2, dash='dash')
    ))

    fig.add_shape(
        type='line',
        x0=str(fecha_corte.date()), x1=str(fecha_corte.date()),
        y0=0, y1=1, yref='paper',
        line=dict(color='gray', dash='dot', width=1.5)
    )
    fig.add_annotation(
        x=str(fecha_corte.date()), y=1, yref='paper',
        text='Corte train/test', showarrow=False,
        font=dict(color='gray', size=11), xanchor='left'
    )

    mae_serie = mean_absolute_error(test_serie['sales'], test_serie['pred_final'])
    titulo_periodo = f"{test_serie['date'].min().strftime('%d %b %Y')} → {test_serie['date'].max().strftime('%d %b %Y')}"

    fig.update_layout(
        title=f'{item} | {store} — {titulo_periodo} | MAE: {mae_serie:.3f}',
        xaxis_title='Fecha',
        yaxis_title='Unidades vendidas',
        template='plotly_white',
        height=400
    )
    fig.show()

In [46]:
productos = [
    ('NYC_1', 'SUPERMARKET_3_090'),
    ('NYC_1', 'SUPERMARKET_3_586'),
    ('NYC_1', 'SUPERMARKET_3_252'),
    ('NYC_1', 'HOME_&_GARDEN_1_118'),
    ('NYC_1', 'HOME_&_GARDEN_1_535'),
    ('NYC_1', 'HOME_&_GARDEN_1_334'),
    ('NYC_1', 'ACCESORIES_1_348'),
    ('NYC_1', 'ACCESORIES_1_371'),
    ('NYC_1', 'ACCESORIES_1_268'),
]

for store, item in productos:
    plot_prediccion_producto(df_train, df_resultados, store, item)

Las tres categorías muestran un patrón claro: cuanto más granular es la serie, más difícil es predecir con precisión.

* **SUPERMARKET** — productos de alta frecuencia y demanda estable. El modelo captura bien el nivel medio y el ritmo semanal. Los errores provienen de picos puntuales no anticipados, no de deriva sistemática.

* **HOME_&_GARDEN y ACCESSORIES** — demanda intermitente: muchos días con ventas bajas o nulas y picos puntuales imposibles de anticipar. El modelo aprende el nivel base correctamente pero suaviza los extremos, subestimando los picos.

Este comportamiento es inherente al problema: a nivel producto individual el WMAPE ronda el 68%, pero al agregar por tienda baja al 8.5% y al nivel total de la cadena al 5.6%. La cancelación de errores al agregar es la razón por la que las predicciones a nivel negocio son mucho más fiables que a nivel producto — y por la que la estrategia bottom-up tiene sentido operativo.

---
## **8. Exportación de resultados**

Guardamos los cuatro niveles de agregación en Parquet para su consumo desde la API de MLOps, y un JSON estructurado por tienda y fecha para facilitar consultas rápidas.

In [47]:
# --- Parquet: cuatro niveles ---
df_nivel_item.to_parquet(DATA_PATH + 'pred_item_tienda.parquet', index=False)
df_nivel_tienda.to_parquet(DATA_PATH + 'pred_tienda.parquet', index=False)
df_nivel_ciudad.to_parquet(DATA_PATH + 'pred_ciudad.parquet', index=False)
df_nivel_total.to_parquet(DATA_PATH + 'pred_total.parquet', index=False)
print('✅ Parquet guardados.')

# --- JSON estructurado por tienda → fecha → métricas ---
import json

json_output = {}
for store in df_nivel_tienda['store_code'].unique():
    df_s = df_nivel_tienda[df_nivel_tienda['store_code'] == store].copy()
    json_output[store] = {
        row['date'].strftime('%Y-%m-%d'): {
            'real':       int(row['sales']),
            'prediccion': round(float(row['pred_final']), 2)
        }
        for _, row in df_s.iterrows()
    }

with open(DATA_PATH + 'predicciones_api.json', 'w') as f:
    json.dump(json_output, f, indent=2)

print('✅ JSON guardado.')
print(f'\nEjemplo — NYC_1, primer día:')
primera_fecha = list(json_output['NYC_1'].keys())[0]
print(f'  {primera_fecha}: {json_output["NYC_1"][primera_fecha]}')

✅ Parquet guardados.
✅ JSON guardado.

Ejemplo — NYC_1, primer día:
  2016-03-28: {'real': 3905, 'prediccion': 4120.03}


---
## **9. Predicciones futuras**

Hasta ahora el modelo se evaluó contra un test interno (Mar-Abr 2016). Esta sección genera predicciones reales para los **28 días siguientes al fin del dataset** (25 Apr → 22 May 2016), cargando los forecasters ya entrenados y actualizando su ventana de contexto con los datos del test para que la predicción arranque desde el punto correcto.

### **9.1 Features exógenas para el período futuro**

Construimos el dataframe de predicción con las mismas features que el modelo vio en entrenamiento. Las rolling features se calculan usando solo datos reales (train + test). Los snapshots con fechas en el futuro se rellenan con la media móvil reciente de cada serie — práctica estándar en M5 para evitar el sesgo de XGBoost al rutear NaN.

In [17]:
# 1. FECHAS FUTURAS Y CALENDARIO
fecha_inicio_futuro = df_test['date'].max() + pd.Timedelta(days=1)
fechas_futuras = pd.date_range(start=fecha_inicio_futuro, periods=28, freq='D')
print(f'Predicción futura: {fechas_futuras[0].date()} → {fechas_futuras[-1].date()}')

df_futuro_cal = pd.DataFrame({'date': fechas_futuras})
df_futuro_cal['dayofweek']         = df_futuro_cal['date'].dt.dayofweek
df_futuro_cal['dayofmonth']        = df_futuro_cal['date'].dt.day
df_futuro_cal['weekofyear']        = df_futuro_cal['date'].dt.isocalendar().week.astype(int)
df_futuro_cal['month']             = df_futuro_cal['date'].dt.month
df_futuro_cal['year']              = df_futuro_cal['date'].dt.year
df_futuro_cal['is_weekend']        = (df_futuro_cal['date'].dt.dayofweek >= 5).astype(int)
df_futuro_cal['dias_desde_inicio'] = (df_futuro_cal['date'] - df_train['date'].min()).dt.days.astype(np.int16)

def get_season(m):
    return 'winter' if m in [12,1,2] else 'spring' if m in [3,4,5] else 'summer' if m in [6,7,8] else 'fall'
def get_pay_period(d):
    return 'start' if d<=7 else 'mid_early' if d<=15 else 'mid_late' if d<=23 else 'end'

df_futuro_cal['season']     = df_futuro_cal['month'].apply(get_season).astype('category')
df_futuro_cal['pay_period'] = df_futuro_cal['dayofmonth'].apply(get_pay_period).astype('category')

years_fut = range(fechas_futuras.year.min(), fechas_futuras.year.max() + 2)
all_holidays_fut = pd.DatetimeIndex(sorted([
    *[pd.Timestamp(f'{y}-{mmdd}') for y in years_fut for mmdd in HOLIDAYS_FIXED_MMDD],
    *[pd.Timestamp(compute_easter(y)) for y in years_fut]
]))
df_futuro_cal['is_holiday']      = df_futuro_cal['date'].isin(all_holidays_fut).astype(int)
df_futuro_cal['days_to_holiday'] = df_futuro_cal['date'].apply(
    lambda f: (all_holidays_fut[all_holidays_fut >= f][0] - f).days
    if len(all_holidays_fut[all_holidays_fut >= f]) > 0 else 0
).astype(np.int16)

# 2. CONSTRUCCIÓN DEL DATAFRAME
ultimo_precio = (
    df_test.sort_values('date')
    .groupby(['item', 'store_code'])[['sell_price', 'indice_estacional_store_item']]
    .last().reset_index()
)

df_futuro = ultimo_precio.merge(df_futuro_cal, how='cross')
df_futuro = df_futuro.merge(df_clusters[['item', 'cluster_kmeans']], on='item', how='left')
df_futuro['cluster_kmeans'] = df_futuro['cluster_kmeans'].astype('category')
df_futuro['id'] = df_futuro['item'].astype(str) + '_' + df_futuro['store_code'].astype(str)
for col in ['store_code', 'item']:
    mediana = df_train.groupby(col)['sales'].median()
    df_futuro[f'te_{col}'] = df_futuro[col].map(mediana).astype(np.float32)

# 3. ROLLING FEATURES DESDE DATOS REALES
df_hist = pd.concat([
    df_train[['id','store_code','item','date','sales']],
    df_test[['id','store_code','item','date','sales']]
]).sort_values(['id','date'])

df_fut_slim = df_futuro[['id','store_code','item','date']].copy()
df_fut_slim['sales'] = np.nan
df_extended = pd.concat([df_hist, df_fut_slim]).sort_values(['id','date']).reset_index(drop=True)

for window in [7, 14, 28]:
    df_extended[f'rmean_{window}_lag28'] = (
        df_extended.groupby('id')['sales']
        .transform(lambda x, w=window: x.shift(28).rolling(w, min_periods=1).mean())
        .astype(np.float32)
    )
df_extended['rstd_7_lag28'] = (
    df_extended.groupby('id')['sales']
    .transform(lambda x: x.shift(28).rolling(7, min_periods=1).std())
    .astype(np.float32)
)
for lag in [7, 14, 21]:
    df_extended[f'snap_lag{lag}'] = (
        df_extended.groupby('id')['sales']
        .transform(lambda x, l=lag: x.shift(l))
        .astype(np.float32)
    )

store_ext = df_extended.groupby(['store_code','date'])['sales'].mean().reset_index(name='ss')
for window in [7, 28]:
    store_ext[f'store_rmean_{window}_lag28'] = (
        store_ext.groupby('store_code')['ss']
        .transform(lambda x, w=window: x.shift(28).rolling(w, min_periods=1).mean())
        .astype(np.float32)
    )
df_extended = df_extended.merge(
    store_ext[['store_code','date','store_rmean_7_lag28','store_rmean_28_lag28']],
    on=['store_code','date'], how='left'
)

rolling_cols = [
    'rmean_7_lag28', 'rmean_14_lag28', 'rmean_28_lag28', 'rstd_7_lag28',
    'snap_lag7', 'snap_lag14', 'snap_lag21',
    'store_rmean_7_lag28', 'store_rmean_28_lag28'
]
rolling_fut = df_extended[df_extended['date'].isin(fechas_futuras)][['id','date'] + rolling_cols]
df_futuro = df_futuro.drop(columns=[c for c in rolling_cols if c in df_futuro.columns], errors='ignore')
df_futuro = df_futuro.merge(rolling_fut, on=['id','date'], how='left')
df_futuro['sales'] = 0.0

# 4. CORRECCIÓN DE NaN EN SNAP FEATURES
# Los snap_lag7/14/21 son NaN para fechas futuras > lag días.
# XGBoost rutea NaN hacia ramas de ventas altas, inflando las predicciones.
# Solución estándar M5: rellenar con la media móvil reciente (rmean_7_lag28).
for lag in [7, 14, 21]:
    col = f'snap_lag{lag}'
    df_futuro[col] = df_futuro[col].fillna(df_futuro['rmean_7_lag28']).astype(np.float32)

del df_extended, df_hist
gc.collect()
print('✅ df_futuro construido:', df_futuro.shape)
print(df_futuro[rolling_cols].isnull().sum())

Predicción futura: 2016-04-25 → 2016-05-22
✅ df_futuro construido: (853720, 30)
rmean_7_lag28           0
rmean_14_lag28          0
rmean_28_lag28          0
rstd_7_lag28            0
snap_lag7               0
snap_lag14              0
snap_lag21              0
store_rmean_7_lag28     0
store_rmean_28_lag28    0
dtype: int64


### **9.2 Predicción**

Cargamos los forecasters entrenados sobre `df_train` y les pasamos el `last_window` con los últimos 42 días de train + test, de modo que el modelo predice desde Apr 25 en lugar de desde el corte original de training.

In [18]:
predicciones_futuras = []

for store in sorted(df_futuro['store_code'].unique()):
    print(f'→ Prediciendo {store}')
    forecaster_fut = joblib.load(DATA_PATH + f'forecaster_{store}.pkl')

    series_validas = list(forecaster_fut.series_names_in_)
    max_lag        = max(forecaster_fut.lags)

    # Solo últimos max_lag días de train (evita pivotar todo df_train)
    fecha_min_ventana = df_train['date'].max() - pd.Timedelta(days=max_lag - 1)
    df_train_slim = df_train[df_train['date'] >= fecha_min_ventana]
    series_train_slim, _ = preparar_datos_tienda(df_train_slim, store, EXOG_FEATURES)
    series_test_store, _ = preparar_datos_tienda(df_test, store, EXOG_FEATURES)

    series_full_store = pd.concat([
        series_train_slim[series_validas],
        series_test_store[series_validas]
    ])
    last_window = series_full_store.tail(max_lag)

    _, exog_fut = preparar_datos_tienda(
        df_futuro[df_futuro['store_code'] == store], store, EXOG_FEATURES
    )
    exog_fut_fil = {k: v for k, v in exog_fut.items() if k in series_validas}

    pred = forecaster_fut.predict(steps=28, exog=exog_fut_fil, last_window=last_window)
    pred_long = pred.reset_index()
    pred_long.columns = ['date', 'item', 'pred_final']
    pred_long['pred_final'] = pred_long['pred_final'].clip(lower=0).round(2)
    pred_long['store_code'] = store
    predicciones_futuras.append(pred_long)
    print(f'  ✅ {store} OK')

df_predicciones_futuras = pd.concat(predicciones_futuras).reset_index(drop=True)
print(f'\nShape: {df_predicciones_futuras.shape}')
print(f'Fechas: {df_predicciones_futuras["date"].min().date()} → {df_predicciones_futuras["date"].max().date()}')
df_predicciones_futuras.to_parquet(DATA_PATH + 'predicciones_futuras.parquet', index=False)
print('✅ Guardado.')

→ Prediciendo BOS_1
  ✅ BOS_1 OK
→ Prediciendo BOS_2
  ✅ BOS_2 OK
→ Prediciendo BOS_3
  ✅ BOS_3 OK
→ Prediciendo NYC_1
  ✅ NYC_1 OK
→ Prediciendo NYC_2
  ✅ NYC_2 OK
→ Prediciendo NYC_3
  ✅ NYC_3 OK
→ Prediciendo NYC_4
  ✅ NYC_4 OK
→ Prediciendo PHI_1
  ✅ PHI_1 OK
→ Prediciendo PHI_2
  ✅ PHI_2 OK
→ Prediciendo PHI_3
  ✅ PHI_3 OK

Shape: (853496, 4)
Fechas: 2016-04-25 → 2016-05-22
✅ Guardado.


### **9.3 Visualización**

 *últimos 28 días reales + predicción de los 28 días siguientes.*

In [19]:
store_ciudad = {
    'NYC_1':'New York','NYC_2':'New York','NYC_3':'New York','NYC_4':'New York',
    'BOS_1':'Boston','BOS_2':'Boston','BOS_3':'Boston',
    'PHI_1':'Philadelphia','PHI_2':'Philadelphia','PHI_3':'Philadelphia'
}

hist_tienda = df_resultados.groupby(['date','store_code'])['sales'].sum().reset_index()
fut_tienda  = df_predicciones_futuras.groupby(['date','store_code'])['pred_final'].sum().reset_index()

for store in sorted(fut_tienda['store_code'].unique()):
    hist = hist_tienda[hist_tienda['store_code'] == store]
    fut  = fut_tienda[fut_tienda['store_code'] == store]
    fig  = go.Figure()
    fig.add_trace(go.Scatter(x=hist['date'], y=hist['sales'],
        name='Real (test)', line=dict(color='#2ca02c', width=2)))
    fig.add_trace(go.Scatter(x=fut['date'], y=fut['pred_final'],
        name='Predicción futura', line=dict(color='#ff7f0e', width=2.5, dash='dash')))
    fig.add_vline(x=str(hist['date'].max().date()), line=dict(color='gray', dash='dot', width=1.5))
    fig.update_layout(title=f'{store} — Predicción futura 28 días',
        template='plotly_white', height=350)
    fig.show()

Las predicciones muestran un nivel coherente con el test y la estructura semanal característica (picos fin de semana, valles entre semana). El arranque ligeramente por debajo del último día del test es esperado — Apr 24 era domingo (pico) y Apr 25 es lunes (valle natural del ciclo semanal).

---
## **10. Conclusiones**

Resumen de resultados, interpretación técnica y traducción al lenguaje del negocio.

### **Resultados técnicos**

**A nivel producto × tienda:**

| Métrica | Benchmark | Modelo | Mejora |
|---|---|---|---|
| MAE | 1.38 | 0.94 | 31.8% |
| WMAPE | 99.6% | 67.9% | 31.8% |

**La mejora aumenta conforme se agrega la jerarquía:**

| Nivel | WMAPE Benchmark | WMAPE Modelo | Mejora |
|---|---|---|---|
| Producto × Tienda | 99.6% | 67.9% | 31.8% |
| Categoría × Tienda | 16.3% | 9.7% | 40.4% |
| Tienda | 14.3% | 8.4% | 41.3% |
| Ciudad | 13.1% | 6.6% | 49.4% |
| Total DSMarket | 12.6% | 5.7% | 54.7% |

### **Interpretación**

El modelo supera al benchmark en todos los niveles jerárquicos sin excepción. A **nivel producto × tienda la mejora es del 31.8%**, y aumenta conforme se agrega la jerarquía — al **nivel total de la cadena** alcanza el **54.7% en WMAPE**.

Esto se explica porque el benchmark estacional simple funciona razonablemente bien capturando el nivel medio por producto, pero falla en los patrones sistemáticos compartidos entre series: el ciclo semanal, el efecto de festivos y la estacionalidad mensual. Nuestro modelo aprende estos patrones de forma global entrenando todas las series de una tienda simultáneamente, y esa ventaja se amplifica al agregar.

La única limitación estructural identificada es la subestimación en los últimos días del horizonte de 28 días — consecuencia inherente del forecasting recursivo con features de ventana deslizante, documentada en soluciones top de la competición M5 (Makridakis, Spiliotis & Assimakopoulos, 2022).

### **Mensaje para el negocio**

Para una tienda con ventas medias de 4.000 unidades diarias, el error de predicción pasa de aproximadamente **570 unidades** (benchmark, WMAPE 14.3%) a **340 unidades** (modelo, WMAPE 8.4%) — una reducción del 41%.

A nivel de cadena completa, el modelo predice el total diario de DSMarket con un error del 5.7% frente al 12.6% del enfoque actual. En términos de planificación financiera y reposición de stock, esto representa una reducción de más del 50% en el error de las estimaciones agregadas que maneja el Director Financiero.

El pipeline es escalable — cada tienda tiene su propio modelo con los mismos hiperparámetros optimizados, extensible a nuevas ubicaciones sin rediseño.